In [1]:
import os, gzip
import urllib.request
import numpy as np
import jax.numpy as jnp
import equinox as eqx
import optax
import jax

MNIST_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"

FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}

def _download(filename, data_dir):
    path = os.path.join(data_dir, filename)
    if not os.path.exists(path):
        urllib.request.urlretrieve(MNIST_URL + filename, path)
    return path

def _read_idx_images(path):
    with gzip.open(path, "rb") as f: data = f.read()
    num_images, rows, cols = np.frombuffer(data, dtype=">u4", count=3, offset=4)
    images = np.frombuffer(data, dtype=np.uint8, offset=16)
    return images.reshape(num_images, rows, cols)

def _read_idx_labels(path):
    with gzip.open(path, "rb") as f: data = f.read()
    return np.frombuffer(data, dtype=np.uint8, offset=8)

def load_mnist(data_dir="./mnist_data"):
    os.makedirs(data_dir, exist_ok=True)

    paths = {key: _download(fname, data_dir) for key, fname in FILES.items()}

    train_images = _read_idx_images(paths["train_images"])
    train_labels = _read_idx_labels(paths["train_labels"])
    test_images = _read_idx_images(paths["test_images"])
    test_labels = _read_idx_labels(paths["test_labels"])

    return (
        jnp.asarray(train_images, dtype=jnp.float32).reshape(train_images.shape[0], -1) / 255.0, 
        jnp.asarray(train_labels, dtype=jnp.int32), 
        jnp.asarray(test_images, dtype=jnp.float32).reshape(test_images.shape[0], -1) / 255.0, 
        jnp.asarray(test_labels, dtype=jnp.int32),
    )

In [2]:
X_train, y_train, X_test, y_test = load_mnist()
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(60000, 784) (60000,) (10000, 784) (10000,)


In [3]:
key = jax.random.PRNGKey(0)
model_key, shuffle_key = jax.random.split(key)

model = eqx.nn.MLP(
    in_size=X_train.shape[1],
    out_size=10,
    width_size=128,
    depth=1,
    activation=jax.nn.tanh,
    key=model_key,
)

optimizer = optax.adam(1e-3)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def loss_fn(model, x, y):
    logits = jax.vmap(model)(x)
    return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()

@eqx.filter_jit
def train_step(model, opt_state, x, y):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

@eqx.filter_jit
def accuracy(model, x, y):
    logits = jax.vmap(model)(x)
    return jnp.mean(jnp.argmax(logits, axis=-1) == y)

batch_size = 128
num_epochs = 10
num_train = X_train.shape[0]
num_batches = num_train // batch_size

for epoch in range(num_epochs):
    shuffle_key, subkey = jax.random.split(shuffle_key)
    perm = jax.random.permutation(subkey, num_train)
    Xtr_shuf, ytr_shuf = X_train[perm], y_train[perm]

    epoch_loss = 0.0
    for i in range(num_batches):
        xb = Xtr_shuf[i * batch_size:(i + 1) * batch_size]
        yb = ytr_shuf[i * batch_size:(i + 1) * batch_size]
        model, opt_state, loss = train_step(model, opt_state, xb, yb)
        epoch_loss += loss

    epoch_loss /= num_batches
    test_acc = accuracy(model, X_test, y_test)
    print(f"epoch {epoch+1:2d} | loss {epoch_loss:.4f} | test acc {test_acc:.4f}")

epoch  1 | loss 0.4111 | test acc 0.9363
epoch  2 | loss 0.1974 | test acc 0.9523
epoch  3 | loss 0.1456 | test acc 0.9612
epoch  4 | loss 0.1136 | test acc 0.9651
epoch  5 | loss 0.0919 | test acc 0.9691
epoch  6 | loss 0.0762 | test acc 0.9714
epoch  7 | loss 0.0637 | test acc 0.9760
epoch  8 | loss 0.0532 | test acc 0.9761
epoch  9 | loss 0.0445 | test acc 0.9774
epoch 10 | loss 0.0378 | test acc 0.9768


In [4]:
from decoupling import Algorithm
from decoupling.scaler import JacobianScaler
from decoupling.utils import collect_information_from_inputs

In [5]:
N = 512

X_dec = Xtr_shuf[:N]

X_dec, Y_dec, J = collect_information_from_inputs(model, X_dec)
X_dec.shape, Y_dec.shape, J.shape

((512, 784), (512, 10), (10, 784, 512))

In [6]:
scaler = JacobianScaler(J)
Js, Ys = scaler.scale(J, Y_dec)

In [13]:
algorithm = Algorithm(rank=8, key=key, niters=10)
decoupling = algorithm.run(X_dec, Ys, Js)
decoupling = scaler.unscale(decoupling)

[Seed 1/1]: 100%|████████████████████████| 10/10 [00:03<00:00,  2.96it/s, error=0.5288, best=0.5288 (9), rcond=1.1e-01]


In [14]:
from typing import Callable

class Linear(eqx.Module):
    weight: jax.Array
    bias: jax.Array

    def __init__(self, weights):
        self.weight = jnp.copy(weights)
        self.bias = jnp.zeros((weights.shape[1],))

    def __call__(self, x):
        print(x.shape)
        print(self.weight.shape)
        print(self.bias.shape)
        return (x @ self.weight) + self.bias

class NN(eqx.Module):
    fc1: Linear
    fc2: Linear
    act: Callable
    
    def __init__(self, decoupling):
        self.fc1 = Linear(decoupling.V)
        self.fc2 = Linear(decoupling.W.T)
        self.act = decoupling.internals

    def __call__(self, x):
        out = self.fc1(x)
        out = self.act(out)
        out = self.fc2(out)
        return out

In [15]:
compressed = NN(decoupling)
compressed

NN(
  fc1=Linear(weight=f32[784,8], bias=f32[8]),
  fc2=Linear(weight=f32[8,10], bias=f32[10]),
  act=<bound method Decoupling.internals of <decoupling.result.result.Decoupling object at 0x7f4d70df6810>>
)

In [16]:
filter_spec = jax.tree_util.tree_map(lambda _: False, compressed)
filter_spec = eqx.tree_at(
    lambda m: (m.fc1.weight, m.fc2.weight),
    filter_spec,
    replace=(True, True),
)

params, static = eqx.partition(compressed, filter_spec)

optimizer = optax.adam(1e-4)
opt_state = optimizer.init(params)

T = 2.0  # distillation temperature

def distill_loss(params, static, x, teacher_logits):
    student = eqx.combine(params, static)
    student_logits = jax.vmap(student)(x)

    teacher_probs = jax.nn.softmax(teacher_logits / T, axis=-1)
    student_log_probs = jax.nn.log_softmax(student_logits / T, axis=-1)

    # KL(teacher || student), scaled by T^2 (Hinton et al.)
    return -jnp.mean(jnp.sum(teacher_probs * student_log_probs, axis=-1)) * (T ** 2)

@eqx.filter_jit
def train_step(params, static, opt_state, x):
    teacher_logits = jax.lax.stop_gradient(jax.vmap(model)(x))
    loss, grads = eqx.filter_value_and_grad(distill_loss)(params, static, x, teacher_logits)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = eqx.apply_updates(params, updates)
    return params, opt_state, loss

@eqx.filter_jit
def accuracy(params, static, x, y):
    net = eqx.combine(params, static)
    logits = jax.vmap(net)(x)
    return jnp.mean(jnp.argmax(logits, axis=-1) == y)

batch_size = 128
num_epochs = 20
num_train = X_train.shape[0]
num_batches = num_train // batch_size

shuffle_key = jax.random.PRNGKey(1)

test_acc = accuracy(params, static, X_test, y_test)
print(f"epoch 0| distill loss {epoch_loss:.4f} | test acc {test_acc:.4f}")

for epoch in range(num_epochs):
    shuffle_key, subkey = jax.random.split(shuffle_key)
    perm = jax.random.permutation(subkey, num_train)
    X_shuf = X_train[perm]

    epoch_loss = 0.0
    for i in range(num_batches):
        xb = X_shuf[i * batch_size:(i + 1) * batch_size]
        params, opt_state, loss = train_step(params, static, opt_state, xb)
        epoch_loss += loss

    epoch_loss /= num_batches
    test_acc = accuracy(params, static, X_test, y_test)
    print(f"epoch {epoch+1:2d} | distill loss {epoch_loss:.4f} | test acc {test_acc:.4f}")

compressed = eqx.combine(params, static)  # fine-tuned student, V and W updated

(784,)
(784, 8)
(8,)
(8,)
(8, 10)
(10,)
epoch 0| distill loss 1.6465 | test acc 0.8012
(784,)
(784, 8)
(8,)
(8,)
(8, 10)
(10,)
epoch  1 | distill loss 2.6207 | test acc 0.9029
epoch  2 | distill loss 2.3170 | test acc 0.9095
epoch  3 | distill loss 2.2466 | test acc 0.9119
epoch  4 | distill loss 2.2133 | test acc 0.9153
epoch  5 | distill loss 2.1923 | test acc 0.9177
epoch  6 | distill loss 2.1770 | test acc 0.9189
epoch  7 | distill loss 2.1671 | test acc 0.9193
epoch  8 | distill loss 2.1588 | test acc 0.9198
epoch  9 | distill loss 2.1511 | test acc 0.9211
epoch 10 | distill loss 2.1444 | test acc 0.9206
epoch 11 | distill loss 2.1391 | test acc 0.9219
epoch 12 | distill loss 2.1341 | test acc 0.9208
epoch 13 | distill loss 2.1295 | test acc 0.9216
epoch 14 | distill loss 2.1250 | test acc 0.9213
epoch 15 | distill loss 2.1208 | test acc 0.9218
epoch 16 | distill loss 2.1174 | test acc 0.9232
epoch 17 | distill loss 2.1142 | test acc 0.9221
epoch 18 | distill loss 2.1109 | test ac

In [17]:
compressed(X_test[0]).argmax(), y_test[0]

(784,)
(784, 8)
(8,)
(8,)
(8, 10)
(10,)


(Array(7, dtype=int32), Array(7, dtype=int32))

In [19]:
model(X_test[0]).argmax(), y_test[0]

(Array(7, dtype=int32), Array(7, dtype=int32))